# PyTorch CNN Deep Learning Labs
## Four-Lab Series: Convolutional Neural Networks from First Principles

![PyTorch](https://img.shields.io/badge/PyTorch-2.8.0-EE4C2C?style=flat&logo=pytorch&logoColor=white)
![Python](https://img.shields.io/badge/Python-3.10+-3776AB?style=flat&logo=python&logoColor=white)
![License](https://img.shields.io/badge/License-MIT-green?style=flat)
![IBM AI Engineering](https://img.shields.io/badge/IBM-AI%20Engineering%20Professional%20Certificate-054ADA?style=flat&logo=ibm&logoColor=white)

**Author:** Jack Pumpuni Frimpong-Manso  
**Course:** IBM AI Engineering Professional Certificate — Deep Neural Networks with PyTorch  
**GitHub:** [github.com/pumpuni07](https://github.com/pumpuni07)

---

> This notebook consolidates four end-to-end PyTorch labs covering the foundational building blocks  
> of Convolutional Neural Networks (CNNs). Run all cells top-to-bottom in a single kernel session.

---
## Table of Contents

| # | Chapter | Key Concepts |
|---|---|---|
| 1 | [What is Convolution?](#chapter-1-what-is-convolution) | Kernel, stride, padding, output size formula |
| 2 | [Activation Functions & Max Pooling](#chapter-2-activation-functions--max-pooling) | ReLU, Sigmoid, Tanh, MaxPool2d |
| 3 | [Logistic Regression: Cross Entropy vs Bad Initialization](#chapter-3-logistic-regression-cross-entropy-vs-bad-initialization) | BCELoss, gradient saturation, weight init |
| 4 | [Fashion-MNIST: CNN vs CNN + Batch Normalization](#chapter-4-fashion-mnist-cnn-vs-cnn--batch-normalization) | Multi-class classification, BatchNorm2d |
| — | [Consolidated Summary](#consolidated-summary) | All key formulas and takeaways |

**Estimated run time:** ~15–20 minutes on CPU (Lab 4 dominates due to dataset download and training)

---
## Environment Setup

All four labs share a single installation and import block. Run this once at the start of your session.

In [ ]:
%%time
# Install all required packages — run once per session
%pip install pandas numpy matplotlib scipy --quiet
%pip install torch==2.8.0+cpu torchvision==0.23.0+cpu torchaudio==2.8.0+cpu \
    --index-url https://download.pytorch.org/whl/cpu --quiet

In [ ]:
# ── Shared imports for all four labs ──────────────────────────────────────────
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as dsets
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from scipy import ndimage, misc
from PIL import Image
import time

torch.manual_seed(0)
print(f"PyTorch version : {torch.__version__}")
print(f"NumPy version   : {np.__version__}")
print("All libraries imported successfully. Ready to run all four labs.")

---
<a id="chapter-1-what-is-convolution"></a>

---

# Chapter 1: What is Convolution?
### Kernel mechanics, output size formula, stride and zero padding

**Estimated time:** 25 minutes

#### Learning Objectives
- Describe convolution as a linear operation on image tensors
- Calculate output size using: $M_{new} = \\lfloor(M + 2p - K) / s\\rfloor + 1$
- Apply stride to control spatial downsampling
- Use zero padding to preserve spatial dimensions

---

## Install and Import Libraries

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage, misc

print(f"PyTorch version: {torch.__version__}")
print("All libraries imported successfully.")

---
## 1. Convolution Defined

Convolution is a **linear operation** — similar to a linear equation, dot product, or matrix multiplication — that is specially suited to image analysis because it:
- **Preserves spatial relationships** between neighbouring pixels
- **Requires fewer parameters** than fully-connected layers (parameter sharing)
- **Detects local features** (edges, textures) regardless of their position in the image

### Mathematical progression

| Operation | Formula |
|---|---|
| Linear equation (scalar) | $y = wx + b$ |
| Linear equation (vector) | $\mathbf{y} = \mathbf{w}\mathbf{x} + b$ |
| Matrix multiplication | $\mathbf{y} = \mathbf{w}\mathbf{X} + \mathbf{b}$ |
| **Convolution (tensor)** | $\mathbf{Y} = \mathbf{w} * \mathbf{X} + \mathbf{b}$ |

In convolution, the parameter **w** is called a **kernel** (or filter). It slides across the input image and performs element-wise multiplication followed by summation to produce each output value.

In [ ]:
# Create a 2D convolution object
# in_channels=1 (grayscale image), out_channels=1 (one feature map), kernel_size=3 (3x3 kernel)
conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3)
print("Convolution layer:", conv)
print("\nDefault (random) state dict:")
print(conv.state_dict())

In [ ]:
# Set the kernel to the Sobel-X operator (detects vertical edges)
# Sobel-X kernel:
#  [ 1,  0, -1]
#  [ 2,  0, -2]
#  [ 1,  0, -1]
conv.state_dict()['weight'][0][0] = torch.tensor([[1.0, 0, -1.0],
                                                   [2.0, 0, -2.0],
                                                   [1.0, 0.0, -1.0]])
conv.state_dict()['bias'][0] = 0.0

print("Kernel (Sobel-X):")
print(conv.state_dict()['weight'][0][0])
print("\nBias:", conv.state_dict()['bias'][0].item())

In [ ]:
# Create a dummy 5x5 grayscale image (1 sample, 1 channel, 5 rows, 5 cols)
# Tensor shape: (N, C, H, W) = (1, 1, 5, 5)
image = torch.zeros(1, 1, 5, 5)

# Set the middle column (index 2) to 1 — creates a vertical edge
image[0, 0, :, 2] = 1

print("Input image (5x5):")
print(image[0, 0])

In [ ]:
# Visualise the image
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(image[0, 0].numpy(), cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Input Image (5×5)\nVertical edge at column 2')
axes[0].set_xlabel('Column')
axes[0].set_ylabel('Row')
for i in range(5):
    for j in range(5):
        axes[0].text(j, i, f'{image[0,0,i,j].item():.0f}',
                     ha='center', va='center', color='red', fontsize=12)

kernel_vals = conv.state_dict()['weight'][0][0].detach().numpy()
axes[1].imshow(kernel_vals, cmap='RdBu', vmin=-2, vmax=2)
axes[1].set_title('Sobel-X Kernel (3×3)\nDetects vertical edges')
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, f'{kernel_vals[i,j]:.0f}',
                     ha='center', va='center', color='black', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('convolution_image_kernel.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved.")

In [ ]:
# Apply convolution
z = conv(image)
print("Output feature map z:")
print(z[0, 0].detach())
print(f"\nOutput shape: {z.shape}  →  (N=1, C_out=1, H_out=3, W_out=3)")
print("\nExplanation: The Sobel-X kernel responds strongly to the vertical edge.")
print("Positive values on the left of the edge, negative on the right.")

---
## 2. Determining the Size of Output

For a square input of size **M** and a square kernel of size **K**, the output size is:

$$M_{new} = M - K + 1$$

**Intuition:** The kernel must fit entirely within the image. Starting at position 0, the last valid starting position is at M-K, giving M-K+1 total positions.

In [ ]:
# Demonstrate output size formula
K = 2  # kernel size
M = 4  # image size

conv1 = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=K)
conv1.state_dict()['weight'][0][0] = torch.tensor([[1.0, 1.0],
                                                    [1.0, 1.0]])
conv1.state_dict()['bias'][0] = 0.0

image1 = torch.ones(1, 1, M, M)
print(f"Input image size:  M = {M}")
print(f"Kernel size:       K = {K}")
print(f"Expected output:   M_new = {M} - {K} + 1 = {M - K + 1}")

z1 = conv1(image1)
print(f"\nActual output z1:")
print(z1.detach())
print(f"Actual output shape: {z1.shape[2:4]}  ✓")

In [ ]:
# Visualise output size for different kernel sizes
M_fixed = 8
kernel_sizes = [2, 3, 4, 5]
output_sizes = [M_fixed - k + 1 for k in kernel_sizes]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar([str(k) for k in kernel_sizes], output_sizes, color='steelblue', edgecolor='black')
for bar, size in zip(bars, output_sizes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            str(size), ha='center', va='bottom', fontweight='bold')
ax.set_xlabel('Kernel Size (K)', fontsize=12)
ax.set_ylabel('Output Size', fontsize=12)
ax.set_title(f'Output Size vs Kernel Size (M={M_fixed}, no padding, stride=1)', fontsize=13)
ax.axhline(y=M_fixed, color='red', linestyle='--', alpha=0.5, label=f'Input size = {M_fixed}')
ax.legend()
plt.tight_layout()
plt.savefig('output_size_vs_kernel.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Stride Parameter

**Stride** controls how many pixels the kernel shifts at each step.  
With stride **s**, the output size formula becomes:

$$M_{new} = \left\lfloor \frac{M - K}{s} \right\rfloor + 1$$

A larger stride reduces the output size and is sometimes used as a downsampling alternative to pooling.

In [ ]:
# Stride = 2 example
stride = 2
K = 2
M = 4

conv3 = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=K, stride=stride)
conv3.state_dict()['weight'][0][0] = torch.tensor([[1.0, 1.0],
                                                    [1.0, 1.0]])
conv3.state_dict()['bias'][0] = 0.0

expected = (M - K) // stride + 1
print(f"Input size M={M}, Kernel K={K}, Stride s={stride}")
print(f"Expected output: floor(({M}-{K})/{stride}) + 1 = {expected}")

z3 = conv3(image1)
print(f"\nOutput z3:")
print(z3.detach())
print(f"Actual shape: {z3.shape[2:4]}  ✓")

In [ ]:
# Compare output sizes across different strides
M_fixed = 8
K_fixed = 3
strides = [1, 2, 3, 4]
out_sizes = [(M_fixed - K_fixed) // s + 1 for s in strides]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar([str(s) for s in strides], out_sizes, color='darkorange', edgecolor='black')
for bar, size in zip(bars, out_sizes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            str(size), ha='center', va='bottom', fontweight='bold')
ax.set_xlabel('Stride (s)', fontsize=12)
ax.set_ylabel('Output Size', fontsize=12)
ax.set_title(f'Output Size vs Stride (M={M_fixed}, K={K_fixed}, no padding)', fontsize=13)
plt.tight_layout()
plt.savefig('output_size_vs_stride.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Zero Padding

**Problem:** Repeated convolutions shrink the feature map rapidly. Boundary pixels are also processed far fewer times than central pixels, losing border information.

**Solution:** Add rows and columns of zeros around the image (**zero padding**).

With padding **p**, the effective image size becomes $M' = M + 2p$, and the output size is:

$$M_{new} = \left\lfloor \frac{M + 2p - K}{s} \right\rfloor + 1$$

**'Same' padding** (output size = input size) requires $p = \frac{K-1}{2}$ when stride = 1.

In [ ]:
# Without padding — non-integer result
K = 2
s = 3
M = 4

conv4 = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=K, stride=s)
conv4.state_dict()['weight'][0][0] = torch.tensor([[1.0, 1.0],
                                                    [1.0, 1.0]])
conv4.state_dict()['bias'][0] = 0.0

raw_size = (M - K) / s + 1
print(f"Without padding: ({M}-{K})/{s} + 1 = {raw_size:.4f}  ← non-integer!")
print(f"PyTorch floors this to: {int(raw_size)}")

z4 = conv4(image1)
print(f"\nOutput z4:")
print(z4.detach())
print(f"Shape: {z4.shape[2:4]}")

In [ ]:
# With padding = 1 — recovers more output coverage
padding = 1
conv5 = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=K, stride=s, padding=padding)
conv5.state_dict()['weight'][0][0] = torch.tensor([[1.0, 1.0],
                                                    [1.0, 1.0]])
conv5.state_dict()['bias'][0] = 0.0

M_prime = M + 2 * padding
new_size = (M_prime - K) // s + 1
print(f"With padding={padding}: M' = {M} + 2×{padding} = {M_prime}")
print(f"Output size = ({M_prime}-{K})//{s} + 1 = {new_size}")

z5 = conv5(image1)
print(f"\nOutput z5:")
print(z5.detach())
print(f"Shape: {z5.shape[2:4]}  ✓")

In [ ]:
# Summary visualisation: how padding and stride interact with output size
M_fixed = 6
K_fixed = 3
configs = [
    {'stride': 1, 'padding': 0, 'label': 's=1, p=0'},
    {'stride': 1, 'padding': 1, 'label': 's=1, p=1 (same)'},
    {'stride': 2, 'padding': 0, 'label': 's=2, p=0'},
    {'stride': 2, 'padding': 1, 'label': 's=2, p=1'},
]
sizes = [(M_fixed + 2*c['padding'] - K_fixed) // c['stride'] + 1 for c in configs]

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['steelblue', 'green', 'darkorange', 'purple']
bars = ax.bar([c['label'] for c in configs], sizes, color=colors, edgecolor='black')
for bar, size in zip(bars, sizes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            str(size), ha='center', va='bottom', fontweight='bold', fontsize=13)
ax.axhline(y=M_fixed, color='red', linestyle='--', alpha=0.6, label=f'Input size = {M_fixed}')
ax.set_ylabel('Output Size', fontsize=12)
ax.set_title(f'Output Size: Effect of Stride & Padding (M={M_fixed}, K={K_fixed})', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig('stride_padding_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Practice Questions

### Q1: Kernel of zeros applied to a random image
**Question:** A kernel of all zeros (kernel_size=3) is applied to a 4×4 random image. What are the output values?

In [ ]:
# Q1: Kernel of all zeros
torch.manual_seed(42)
Image_q1 = torch.randn((1, 1, 4, 4))
print("Random input image:")
print(Image_q1[0, 0])

conv_q1 = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3)
conv_q1.state_dict()['weight'][0][0] = torch.tensor([[0.0, 0.0, 0.0],
                                                      [0.0, 0.0, 0.0],
                                                      [0.0, 0.0, 0.0]])
conv_q1.state_dict()['bias'][0] = 0.0

z_q1 = conv_q1(Image_q1)
print("\nOutput (zero kernel):")
print(z_q1.detach())
print("\n✓ Answer: All output values are ZERO.")
print("  Reason: Every element-wise product with a zero kernel is 0.")
print(f"  Output shape: {z_q1.shape[2:]} = M - K + 1 = 4 - 3 + 1 = 2")

### Q2: Output size with kernel_size=2, stride=2

In [ ]:
# Q2: Compute output size analytically and verify
M_q2 = 4
K_q2 = 2
s_q2 = 2

M_new_q2 = (M_q2 - K_q2) // s_q2 + 1
print(f"Image size M = {M_q2}, Kernel K = {K_q2}, Stride s = {s_q2}")
print(f"Output size = ({M_q2} - {K_q2}) // {s_q2} + 1 = {M_new_q2}")

conv_q2 = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=K_q2, stride=s_q2)
conv_q2.state_dict()['weight'][0][0] = torch.tensor([[1.0, 1.0],
                                                      [1.0, 1.0]])
conv_q2.state_dict()['bias'][0] = 0.0
image_q2 = torch.ones(1, 1, M_q2, M_q2)
z_q2 = conv_q2(image_q2)
print(f"\nVerification — actual output shape: {z_q2.shape[2:]}  ✓")
print(f"Output values: {z_q2.detach()}")

---
## 6. Summary

| Concept | Formula | Key Takeaway |
|---|---|---|
| Basic output size | $M_{new} = M - K + 1$ | Output is smaller than input |
| With stride | $M_{new} = \lfloor\frac{M-K}{s}\rfloor + 1$ | Larger stride → smaller output |
| With padding | $M_{new} = \lfloor\frac{M+2p-K}{s}\rfloor + 1$ | Padding recovers lost size |
| Same padding (s=1) | $p = \frac{K-1}{2}$ | Output = Input size |

In [ ]:
# Final comprehensive summary demo
def conv_output_size(M, K, s=1, p=0):
    """Calculate convolution output size given M, K, stride s, and padding p."""
    return (M + 2*p - K) // s + 1

print("=" * 55)
print("Convolution Output Size Calculator")
print("=" * 55)
test_cases = [
    (5, 3, 1, 0),  # basic
    (4, 2, 1, 0),  # basic
    (4, 2, 2, 0),  # stride
    (4, 2, 3, 1),  # stride + padding
    (6, 3, 1, 1),  # same padding
]
for M, K, s, p in test_cases:
    out = conv_output_size(M, K, s, p)
    print(f"M={M}, K={K}, s={s}, p={p}  →  output size = {out}")

---
<a id="chapter-2-activation-functions--max-pooling"></a>

---

# Chapter 2: Activation Functions & Max Pooling
### Non-linearity with ReLU and spatial downsampling with MaxPool2d

**Estimated time:** 25 minutes

#### Learning Objectives
- Apply ReLU activation element-wise to convolutional feature maps
- Explain why non-linearity is essential in deep networks
- Implement max pooling with overlapping and non-overlapping windows
- Trace spatial dimensions through a Conv → ReLU → MaxPool pipeline

---

## Install and Import Libraries

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage, misc

print(f"PyTorch version: {torch.__version__}")
print("All libraries imported successfully.")

---
## 1. Activation Functions

### Why do we need activation functions?

A convolution is a **linear** operation. Stacking multiple linear operations gives you another linear operation — no matter how many layers deep, the network could be collapsed into a single layer with no gain in expressive power.

**Activation functions introduce non-linearity**, allowing the network to learn complex, hierarchical patterns.

### ReLU (Rectified Linear Unit)

$$\text{ReLU}(x) = \max(0, x)$$

- All negative values → 0
- Positive values → unchanged
- **Fast to compute**, avoids vanishing gradient (vs. sigmoid/tanh)
- Applied **element-wise** to every value in the activation map

In [ ]:
# Build the convolution layer with a Sobel-X kernel
conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3)
Gx = torch.tensor([[1.0, 0, -1.0],
                    [2.0, 0, -2.0],
                    [1.0, 0, -1.0]])
conv.state_dict()['weight'][0][0] = Gx
conv.state_dict()['bias'][0] = 0.0

print("Sobel-X kernel:")
print(conv.state_dict()['weight'][0][0])

In [ ]:
# Create 5×5 image with a vertical edge
image = torch.zeros(1, 1, 5, 5)
image[0, 0, :, 2] = 1
print("Input image:")
print(image[0, 0])

In [ ]:
# Step 1: Apply convolution
Z = conv(image)
print("Activation map Z (after convolution):")
print(Z[0, 0].detach())
print("\nNotice: negative values present (response to right side of edge)")

In [ ]:
# Step 2: Apply ReLU
A = torch.relu(Z)
print("After ReLU (negative values zeroed out):")
print(A[0, 0].detach())

# Also using the nn.ReLU module (equivalent)
relu = nn.ReLU()
A_module = relu(Z)
print("\nUsing nn.ReLU() module (same result):")
print(A_module[0, 0].detach())

In [ ]:
# Visualise ReLU effect on activation map
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

Z_np = Z[0, 0].detach().numpy()
A_np = A[0, 0].detach().numpy()

im0 = axes[0].imshow(image[0, 0].numpy(), cmap='gray')
axes[0].set_title('Input Image (5×5)\nVertical edge at col 2', fontsize=11)
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(Z_np, cmap='RdBu', vmin=-4, vmax=4)
axes[1].set_title('Activation Map Z\n(after Conv, before ReLU)', fontsize=11)
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, f'{Z_np[i,j]:.0f}',
                     ha='center', va='center', fontsize=13, fontweight='bold')
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(A_np, cmap='Greens')
axes[2].set_title('After ReLU: max(0, Z)\n(negatives → 0)', fontsize=11)
for i in range(3):
    for j in range(3):
        color = 'black' if A_np[i,j] > 0 else 'red'
        axes[2].text(j, i, f'{A_np[i,j]:.0f}',
                     ha='center', va='center', fontsize=13, fontweight='bold', color=color)
plt.colorbar(im2, ax=axes[2])

plt.suptitle('Convolution → Activation Map → ReLU', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('relu_activation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: relu_activation.png")

In [ ]:
# Compare activation functions: ReLU vs Sigmoid vs Tanh
x_vals = torch.linspace(-4, 4, 200)

relu_out   = torch.relu(x_vals)
sigmoid_out = torch.sigmoid(x_vals)
tanh_out   = torch.tanh(x_vals)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].plot(x_vals.numpy(), relu_out.numpy(), 'b-', linewidth=2)
axes[0].set_title('ReLU: max(0, x)', fontsize=12)
axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('x'); axes[0].set_ylabel('f(x)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(x_vals.numpy(), sigmoid_out.numpy(), 'r-', linewidth=2)
axes[1].set_title('Sigmoid: 1/(1+e⁻ˣ)', fontsize=12)
axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('x')
axes[1].grid(True, alpha=0.3)

axes[2].plot(x_vals.numpy(), tanh_out.numpy(), 'g-', linewidth=2)
axes[2].set_title('Tanh: (eˣ - e⁻ˣ)/(eˣ + e⁻ˣ)', fontsize=12)
axes[2].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[2].set_xlabel('x')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Common Activation Functions in CNNs', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('activation_functions_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: activation_functions_comparison.png")

---
## 2. Max Pooling

### What is Max Pooling?

Max pooling is a **spatial downsampling** operation that:
- Divides the feature map into non-overlapping (or overlapping) windows
- Retains only the **maximum value** from each window
- **Reduces spatial dimensions** → fewer parameters, less computation
- Provides **translation invariance**: a feature detected slightly off-position still activates

The output size formula is the same as convolution:
$$M_{new} = \left\lfloor \frac{M - K_{pool}}{s} \right\rfloor + 1$$

In [ ]:
# Create a 4×4 image with known values
image1 = torch.zeros(1, 1, 4, 4)
image1[0, 0, 0, :] = torch.tensor([1.0, 2.0, 3.0, -4.0])
image1[0, 0, 1, :] = torch.tensor([0.0, 2.0, -3.0, 0.0])
image1[0, 0, 2, :] = torch.tensor([0.0, 2.0, 3.0, 1.0])
image1[0, 0, 3, :] = torch.tensor([0.0, 1.0, 1.0, 0.0])

print("Input feature map (4×4):")
print(image1[0, 0])

In [ ]:
# Max Pooling with stride=1 (overlapping windows)
max1 = torch.nn.MaxPool2d(2, stride=1)
result1 = max1(image1)
print("Max Pooling (kernel=2, stride=1) — overlapping windows:")
print(result1[0, 0].detach())
print(f"Output shape: {result1.shape[2:]}")
print(f"  → M_new = floor((4-2)/1) + 1 = 3")

In [ ]:
# Max Pooling with stride=None (defaults to kernel size — non-overlapping)
max2 = torch.nn.MaxPool2d(2)  # stride defaults to kernel_size=2
result2 = max2(image1)
print("Max Pooling (kernel=2, stride=2) — non-overlapping windows:")
print(result2[0, 0].detach())
print(f"Output shape: {result2.shape[2:]}")
print(f"  → M_new = floor((4-2)/2) + 1 = 2")
print()
print("Explanation of each output value:")
print(f"  Top-left 2×2 window:     max(1,2,0,2)  = {max(1,2,0,2)}")
print(f"  Top-right 2×2 window:    max(3,-4,-3,0) = {max(3,-4,-3,0)}")
print(f"  Bottom-left 2×2 window:  max(0,2,0,1)  = {max(0,2,0,1)}")
print(f"  Bottom-right 2×2 window: max(3,1,1,0)  = {max(3,1,1,0)}")

In [ ]:
# Visualise Max Pooling step by step
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Original
img_np = image1[0, 0].numpy()
im0 = axes[0].imshow(img_np, cmap='coolwarm', vmin=-4, vmax=4)
axes[0].set_title('Input Feature Map (4×4)', fontsize=11)
for i in range(4):
    for j in range(4):
        axes[0].text(j, i, f'{img_np[i,j]:.0f}',
                     ha='center', va='center', fontsize=13, fontweight='bold')
plt.colorbar(im0, ax=axes[0])

# Stride=1
r1_np = result1[0, 0].detach().numpy()
im1 = axes[1].imshow(r1_np, cmap='coolwarm')
axes[1].set_title('MaxPool(2, stride=1)\n→ 3×3 output', fontsize=11)
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, f'{r1_np[i,j]:.0f}',
                     ha='center', va='center', fontsize=13, fontweight='bold')
plt.colorbar(im1, ax=axes[1])

# Stride=2
r2_np = result2[0, 0].detach().numpy()
im2 = axes[2].imshow(r2_np, cmap='coolwarm')
axes[2].set_title('MaxPool(2, stride=2)\n→ 2×2 output (non-overlapping)', fontsize=11)
for i in range(2):
    for j in range(2):
        axes[2].text(j, i, f'{r2_np[i,j]:.0f}',
                     ha='center', va='center', fontsize=13, fontweight='bold')
plt.colorbar(im2, ax=axes[2])

plt.suptitle('Max Pooling: Spatial Downsampling', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('max_pooling_demo.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: max_pooling_demo.png")

---
## 3. Combining Conv → ReLU → MaxPool

In a real CNN, these three operations form the **core building block** of each convolutional layer:

```
Input → Conv2d → ReLU → MaxPool2d → (next layer)
```

- **Conv2d:** Detects features (edges, textures, shapes)
- **ReLU:** Introduces non-linearity, suppresses weak responses
- **MaxPool2d:** Reduces spatial size, retains dominant features, provides invariance

In [ ]:
# Build a mini pipeline: Conv → ReLU → MaxPool
torch.manual_seed(0)

# Define a 7×7 test image with a central vertical edge
image_full = torch.zeros(1, 1, 7, 7)
image_full[0, 0, :, 3] = 1.0  # vertical edge at column 3

# Conv layer with Sobel-X
conv_layer = nn.Conv2d(1, 1, kernel_size=3, padding=0)
conv_layer.state_dict()['weight'][0][0] = torch.tensor([[ 1.0,  0.0, -1.0],
                                                         [ 2.0,  0.0, -2.0],
                                                         [ 1.0,  0.0, -1.0]])
conv_layer.state_dict()['bias'][0] = 0.0

# ReLU
relu_layer = nn.ReLU()

# MaxPool
pool_layer = nn.MaxPool2d(2, stride=2)

# Forward pass
Z = conv_layer(image_full)     # Conv: 7→5
A = relu_layer(Z)              # ReLU: shape unchanged
P = pool_layer(A)              # Pool: 5→2

print(f"Input shape:   {image_full.shape}")
print(f"After Conv:    {Z.shape}  (7-3+1=5)")
print(f"After ReLU:    {A.shape}  (shape unchanged)")
print(f"After MaxPool: {P.shape}  (floor((5-2)/2)+1=2)")

print("\nConv output Z:")
print(Z[0, 0].detach())
print("\nAfter ReLU A:")
print(A[0, 0].detach())
print("\nAfter MaxPool P:")
print(P[0, 0].detach())

In [ ]:
# Full pipeline visualisation
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

stages = [
    (image_full[0, 0].numpy(), 'Input (7×7)', 'gray'),
    (Z[0, 0].detach().numpy(),  'Conv Output Z (5×5)', 'RdBu'),
    (A[0, 0].detach().numpy(),  'After ReLU A (5×5)', 'Greens'),
    (P[0, 0].detach().numpy(),  'After MaxPool P (2×2)', 'YlOrRd'),
]

for ax, (data, title, cmap) in zip(axes, stages):
    im = ax.imshow(data, cmap=cmap)
    ax.set_title(title, fontsize=10)
    H, W = data.shape
    for i in range(H):
        for j in range(W):
            ax.text(j, i, f'{data[i,j]:.0f}',
                    ha='center', va='center', fontsize=10,
                    color='black' if abs(data[i,j]) < 3 else 'white')
    plt.colorbar(im, ax=ax)

plt.suptitle('CNN Building Block: Conv → ReLU → MaxPool', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('cnn_pipeline_demo.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: cnn_pipeline_demo.png")

---
## 4. Summary

| Component | Operation | Purpose |
|---|---|---|
| `nn.Conv2d` | Element-wise multiply + sum, sliding kernel | Feature detection (edges, textures) |
| `nn.ReLU` | $\max(0, x)$ applied element-wise | Introduces non-linearity, sparsity |
| `nn.MaxPool2d` | Max value in each pooling window | Downsampling, translation invariance |

### Key formulas recap

| Operation | Output Size |
|---|---|
| Conv (basic) | $M - K + 1$ |
| Conv (stride s, padding p) | $\lfloor(M + 2p - K)/s\rfloor + 1$ |
| MaxPool (kernel K, stride s) | $\lfloor(M - K)/s\rfloor + 1$ |

In [ ]:
# Dimension tracking through a typical CNN block
print("Dimension tracking: Conv → ReLU → MaxPool")
print("=" * 50)

M = 32  # e.g., 32×32 input
K_conv = 5
p_conv = 2  # 'same' padding for k=5
s_conv = 1
K_pool = 2
s_pool = 2

after_conv = (M + 2*p_conv - K_conv) // s_conv + 1
after_relu = after_conv  # shape unchanged
after_pool = (after_relu - K_pool) // s_pool + 1

print(f"Input:          {M}×{M}")
print(f"Conv2d(K={K_conv}, s={s_conv}, p={p_conv}):   {after_conv}×{after_conv}")
print(f"ReLU:           {after_relu}×{after_relu}  (unchanged)")
print(f"MaxPool(K={K_pool}, s={s_pool}):  {after_pool}×{after_pool}")
print(f"\nDownsampling ratio: {M}/{after_pool} = {M/after_pool:.1f}x")

---
<a id="chapter-3-logistic-regression-cross-entropy-vs-bad-initialization"></a>

---

# Chapter 3: Logistic Regression: Cross Entropy vs Bad Initialization
### BCELoss vs MSE, gradient saturation, and the cost of large initial weights

**Estimated time:** 25 minutes

#### Learning Objectives
- Implement binary logistic regression with sigmoid + BCELoss
- Explain why Cross Entropy outperforms MSE for classification tasks
- Demonstrate gradient saturation caused by bad weight initialization
- Compare accuracy: good init (~100%) vs bad init (~60%)

---

## Install and Import Libraries

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(0)
print(f"PyTorch version: {torch.__version__}")

---
## 1. Dataset Setup

We create a synthetic binary classification dataset and a logistic regression model.

In [ ]:
# Dataset class
class BinaryData(Dataset):
    """
    Synthetic binary classification dataset.
    Points with x < 0 are class 0; points with x >= 0 are class 1.
    """
    def __init__(self, n_samples=100):
        torch.manual_seed(1)  # reproducible
        self.x = torch.arange(-3, 3, 6 / n_samples).view(-1, 1).float()
        self.y = torch.zeros(n_samples, 1)
        self.y[n_samples // 2:] = 1.0  # upper half → class 1
        self.len = n_samples

    def __getitem__(self, index):
        return self.x[index], self.y[index]

    def __len__(self):
        return self.len


# Logistic Regression model
class LogisticRegressionModel(nn.Module):
    """Single linear layer followed by sigmoid activation."""
    def __init__(self):
        super(LogisticRegressionModel, self).__init__()
        self.linear = nn.Linear(1, 1)

    def forward(self, x):
        return torch.sigmoid(self.linear(x))


# Create dataset and dataloader
data_set = BinaryData(n_samples=100)
trainloader = DataLoader(dataset=data_set, batch_size=10, shuffle=True)

print(f"Dataset size:      {len(data_set)} samples")
print(f"Input range:       [{data_set.x.min():.2f}, {data_set.x.max():.2f}]")
print(f"Class distribution: {(data_set.y == 0).sum().item()} class-0, "
      f"{(data_set.y == 1).sum().item()} class-1")

In [ ]:
# Visualise the dataset
fig, ax = plt.subplots(figsize=(9, 4))
mask0 = (data_set.y[:, 0] == 0)
mask1 = (data_set.y[:, 0] == 1)
ax.scatter(data_set.x[mask0].numpy(), data_set.y[mask0].numpy(),
           color='blue', label='Class 0', alpha=0.6, edgecolors='k')
ax.scatter(data_set.x[mask1].numpy(), data_set.y[mask1].numpy(),
           color='red',  label='Class 1', alpha=0.6, edgecolors='k')
ax.set_xlabel('x (feature)', fontsize=12)
ax.set_ylabel('y (label)', fontsize=12)
ax.set_title('Binary Classification Dataset', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('binary_dataset.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2. Cross Entropy Loss — Good Initialization

### Why Cross Entropy for classification?

**MSE** on sigmoid outputs suffers from extremely flat gradients when predictions are far from the true label — this causes **gradient saturation** (vanishing gradients).

**Binary Cross Entropy (BCE)** is derived from Maximum Likelihood Estimation and provides strong, consistent gradients:

$$\mathcal{L}_{BCE} = -\frac{1}{N}\sum_{i=1}^{N} \left[ y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i) \right]$$

Good initialization means weights start **close to zero** — sigmoid outputs near 0.5, gradients are healthy.

In [ ]:
def train_model(model, criterion, optimizer, dataloader, epochs=100):
    """Train a model and return per-epoch loss history."""
    loss_history = []
    for epoch in range(epochs):
        epoch_loss = 0.0
        for x_batch, y_batch in dataloader:
            optimizer.zero_grad()
            yhat = model(x_batch)
            loss = criterion(yhat, y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        loss_history.append(epoch_loss / len(dataloader))
    return loss_history


def evaluate(model, dataset):
    """Return accuracy (fraction of correct predictions)."""
    with torch.no_grad():
        yhat = model(dataset.x)
        label = (yhat > 0.5).float()
        correct = (label == dataset.y).float().mean().item()
    return correct

In [ ]:
# ---- GOOD INITIALIZATION ----
torch.manual_seed(0)
model_good = LogisticRegressionModel()

print("Initial weights (good init — near zero):")
print(f"  w = {model_good.linear.weight.item():.4f}")
print(f"  b = {model_good.linear.bias.item():.4f}")

criterion_bce = nn.BCELoss()
optimizer_good = torch.optim.SGD(model_good.parameters(), lr=2.0)

loss_good = train_model(model_good, criterion_bce, optimizer_good, trainloader, epochs=100)
acc_good = evaluate(model_good, data_set)

print(f"\nFinal accuracy (good init): {acc_good * 100:.1f}%")
print(f"Final loss:                 {loss_good[-1]:.4f}")

---
## 3. Cross Entropy Loss — Bad Initialization

**Bad initialization** forces the sigmoid far into saturation at the start of training. When weights are large, outputs approach 0 or 1, and the BCE gradient becomes very small — the model struggles to escape this plateau.

Result: lower accuracy and higher loss at convergence.

In [ ]:
# ---- BAD INITIALIZATION ----
torch.manual_seed(0)
model_bad = LogisticRegressionModel()

# Force large weight values → sigmoid saturates immediately
model_bad.state_dict()['linear.weight'][0] = torch.tensor([-15.0])
model_bad.state_dict()['linear.bias'][0]   = torch.tensor([3.0])

print("Initial weights (bad init — large values push sigmoid to saturation):")
print(f"  w = {model_bad.linear.weight.item():.4f}")
print(f"  b = {model_bad.linear.bias.item():.4f}")

# Sigmoid output at extremes with bad init
x_sample = data_set.x[[0, 50, 99]]  # three representative points
with torch.no_grad():
    sigmoid_out = model_bad(x_sample)
print(f"\nSigmoid outputs at start (x={x_sample.flatten().tolist()}):")
print(f"  {sigmoid_out.flatten().tolist()}")
print("  → Already near 0 or 1 before any training — gradients ≈ 0!")

optimizer_bad = torch.optim.SGD(model_bad.parameters(), lr=2.0)
loss_bad = train_model(model_bad, criterion_bce, optimizer_bad, trainloader, epochs=100)
acc_bad = evaluate(model_bad, data_set)

print(f"\nFinal accuracy (bad init):  {acc_bad * 100:.1f}%")
print(f"Final loss:                 {loss_bad[-1]:.4f}")

---
## 4. Side-by-Side Comparison

In [ ]:
# Loss curves comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Loss curves
axes[0].plot(loss_good, 'b-', linewidth=2, label=f'Good Init (final acc={acc_good*100:.1f}%)')
axes[0].plot(loss_bad,  'r--', linewidth=2, label=f'Bad Init  (final acc={acc_bad*100:.1f}%)')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('BCE Loss', fontsize=12)
axes[0].set_title('Training Loss: Good vs Bad Initialization', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Decision boundary comparison
x_plot = torch.linspace(-3, 3, 300).unsqueeze(1)
with torch.no_grad():
    y_good = model_good(x_plot).numpy()
    y_bad  = model_bad(x_plot).numpy()

axes[1].scatter(data_set.x[mask0].numpy(), data_set.y[mask0].numpy(),
                color='blue', alpha=0.4, label='Class 0', edgecolors='k', s=30)
axes[1].scatter(data_set.x[mask1].numpy(), data_set.y[mask1].numpy(),
                color='red',  alpha=0.4, label='Class 1', edgecolors='k', s=30)
axes[1].plot(x_plot.numpy(), y_good, 'b-', linewidth=2.5, label='Good init sigmoid')
axes[1].plot(x_plot.numpy(), y_bad,  'r--', linewidth=2.5, label='Bad init sigmoid')
axes[1].axhline(0.5, color='gray', linestyle=':', alpha=0.7, label='Decision boundary (0.5)')
axes[1].set_xlabel('x (feature)', fontsize=12)
axes[1].set_ylabel('P(class=1)', fontsize=12)
axes[1].set_title('Learned Decision Boundaries', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Cross Entropy Loss: Good vs Bad Initialization', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('good_vs_bad_init.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: good_vs_bad_init.png")

In [ ]:
# Why BCE beats MSE: gradient magnitude comparison
p_vals = np.linspace(0.01, 0.99, 200)
y_true = 1  # true class = 1

# BCE gradient magnitude: dL/dp = -(y/p) + (1-y)/(1-p)
bce_grad = np.abs(-(y_true / p_vals) + (1 - y_true) / (1 - p_vals))

# MSE gradient magnitude (for σ-activated output): dL/dp = 2(p - y)
mse_grad = np.abs(2 * (p_vals - y_true))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(p_vals, bce_grad, 'b-', linewidth=2.5, label='|∂BCE/∂p|')
ax.plot(p_vals, mse_grad, 'r--', linewidth=2.5, label='|∂MSE/∂p|')
ax.set_xlabel('Predicted probability p', fontsize=12)
ax.set_ylabel('|Gradient magnitude|', fontsize=12)
ax.set_title('BCE vs MSE Gradient Magnitude (y=1, correct → p=1)\n'
             'BCE provides large gradients even when predictions are far off', fontsize=11)
ax.legend(fontsize=12)
ax.set_ylim(0, 10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('bce_vs_mse_gradients.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: bce_vs_mse_gradients.png")

---
## 5. Summary and Key Takeaways

| Aspect | Good Initialization | Bad Initialization |
|---|---|---|
| Initial weights | Small (near zero) | Large (saturated sigmoid) |
| Initial gradients | Healthy | Near zero (vanishing) |
| Training speed | Fast convergence | Slow or stuck |
| Final accuracy | ~100% | ~60% |

### Key Rules
1. **Use `nn.BCELoss`** (or `nn.BCEWithLogitsLoss`) for binary classification — not MSE
2. **Initialize weights near zero** (PyTorch default for `nn.Linear` uses Kaiming uniform, which is generally good)
3. **Avoid manually setting large initial weights** unless you have a specific architectural reason
4. **Batch Normalization** can help mitigate bad initialization by normalizing layer inputs

In [ ]:
# Final results summary
print("=" * 55)
print("FINAL RESULTS SUMMARY")
print("=" * 55)
print(f"{'Metric':<30} {'Good Init':>10} {'Bad Init':>10}")
print("-" * 55)
print(f"{'Accuracy':<30} {acc_good*100:>9.1f}% {acc_bad*100:>9.1f}%")
print(f"{'Final BCE Loss':<30} {loss_good[-1]:>10.4f} {loss_bad[-1]:>10.4f}")
print(f"{'Loss epoch 1':<30} {loss_good[0]:>10.4f} {loss_bad[0]:>10.4f}")
print(f"{'Loss epoch 10':<30} {loss_good[9]:>10.4f} {loss_bad[9]:>10.4f}")

---
<a id="chapter-4-fashion-mnist-cnn-vs-cnn--batch-normalization"></a>

---

# Chapter 4: Fashion-MNIST: CNN vs CNN + Batch Normalization
### 10-class image classification with and without Batch Normalization

**Estimated time:** 30 minutes

#### Learning Objectives
- Build and train a two-layer CNN for multi-class image classification
- Implement Batch Normalization and explain its effect on training stability
- Compare CNN vs CNN_batch training dynamics over 5 epochs
- Perform per-class accuracy analysis and error inspection

---

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as dsets
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from PIL import Image
import time

torch.manual_seed(0)
print(f"PyTorch version: {torch.__version__}")
device = torch.device('cpu')
print(f"Device: {device}")

---
## 1. Environment Setup & Dataset

### Fashion-MNIST Overview
- **60,000 training** + **10,000 test** grayscale images
- **28×28 pixels**, 10 clothing categories
- Direct drop-in replacement for MNIST, but harder

| Label | Class |
|---|---|
| 0 | T-shirt/top |
| 1 | Trouser |
| 2 | Pullover |
| 3 | Dress |
| 4 | Coat |
| 5 | Sandal |
| 6 | Shirt |
| 7 | Sneaker |
| 8 | Bag |
| 9 | Ankle boot |

In [ ]:
# Class names for Fashion-MNIST
CLASS_NAMES = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]
IMAGE_SIZE = 16  # Resize 28×28 → 16×16 for faster training

# Compose transforms: Resize + ToTensor
composed = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor()
])

# Download Fashion-MNIST
dataset_train = dsets.FashionMNIST(
    root='.fashion/data', train=True,  transform=composed, download=True
)
dataset_val = dsets.FashionMNIST(
    root='.fashion/data', train=False, transform=composed, download=True
)

print(f"Training samples:   {len(dataset_train):,}")
print(f"Validation samples: {len(dataset_val):,}")
print(f"Image size after resize: {IMAGE_SIZE}×{IMAGE_SIZE}")
print(f"Image tensor shape: {dataset_train[0][0].shape}")

In [ ]:
# ── show_data helper — displays one Fashion-MNIST sample ────────────────────
def show_data(data_sample, ax=None):
    """Display a single Fashion-MNIST sample with class name and numeric label."""
    img, label = data_sample
    if ax is None:
        fig, ax = plt.subplots(figsize=(3, 3))
    ax.imshow(img.numpy().reshape(IMAGE_SIZE, IMAGE_SIZE), cmap='gray')
    ax.set_title(f'{CLASS_NAMES[label]}\n(y = {label})', fontsize=10, fontweight='bold')
    ax.axis('off')

#### Task 1 — Display First Three Validation Samples

Display the first **three** validation samples individually.  
Each figure shows the resized **16×16** image with its class name and numeric label `y`.

In [ ]:
# ── Task 1: first 3 validation samples — one figure each ────────────────────
# Matches the original lab specification exactly:
#   for n, data_sample in enumerate(dataset_val):
#       show_data(data_sample)
#       plt.show()
#       if n == 2: break

for n, data_sample in enumerate(dataset_val):
    img, label = data_sample
    fig, ax = plt.subplots(figsize=(3, 3))
    ax.imshow(img.numpy().reshape(IMAGE_SIZE, IMAGE_SIZE), cmap='gray')
    ax.set_title(
        f'Sample index : {n}\n'
        f'Class        : {CLASS_NAMES[label]}\n'
        f'Label (y)    : {label}',
        fontsize=10, fontweight='bold', pad=8
    )
    ax.set_xlabel(f'Resized: {IMAGE_SIZE}×{IMAGE_SIZE} px  (original: 28×28)', fontsize=8)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for spine in ax.spines.values():
        spine.set_edgecolor('steelblue')
        spine.set_linewidth(2)
    plt.tight_layout()
    plt.show()
    if n == 2:
        break

#### Extended View — First 9 Validation Samples (Grid)

Broader view of the validation set to appreciate class diversity across 10 categories.

In [ ]:
# ── Extended: first 9 validation samples in a labelled grid ─────────────────
fig, axes = plt.subplots(3, 3, figsize=(8, 8))
for i, ax in enumerate(axes.flatten()):
    show_data(dataset_val[i], ax=ax)
plt.suptitle(
    f'Fashion-MNIST — First 9 Validation Samples\n'
    f'(resized to {IMAGE_SIZE}×{IMAGE_SIZE} from original 28×28)',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig('fashion_mnist_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fashion_mnist_samples.png")

In [ ]:
# DataLoaders
BATCH_SIZE = 100
train_loader = DataLoader(dataset=dataset_train, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(dataset=dataset_val,   batch_size=BATCH_SIZE, shuffle=False)
print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")

---
## 2. Model Architectures

### Architecture overview (16×16 input)

```
Input (1×16×16)
    ↓
Conv2d(1→16, k=5, p=2)  → 16×16×16  [same padding]
[BatchNorm2d(16)]         → optional
ReLU
MaxPool2d(k=2)           → 8×8×16
    ↓
Conv2d(16→32, k=5, p=2) → 8×8×32   [same padding]
[BatchNorm2d(32)]         → optional
ReLU
MaxPool2d(k=2)           → 4×4×32
    ↓
Flatten                  → 32×4×4 = 512
Linear(512 → 10)
[BatchNorm1d(10)]         → optional
    ↓
Output logits (10 classes)
```

In [ ]:
# -----------------------------------------------------------------------
# Standard CNN
# -----------------------------------------------------------------------
class CNN(nn.Module):
    """Two-layer CNN for Fashion-MNIST classification."""

    def __init__(self, out_1=16, out_2=32, number_of_classes=10):
        super(CNN, self).__init__()
        # Block 1
        self.cnn1     = nn.Conv2d(in_channels=1, out_channels=out_1, kernel_size=5, padding=2)
        self.maxpool1 = nn.MaxPool2d(kernel_size=2)
        # Block 2
        self.cnn2     = nn.Conv2d(in_channels=out_1, out_channels=out_2, kernel_size=5,
                                   stride=1, padding=2)
        self.maxpool2 = nn.MaxPool2d(kernel_size=2)
        # Classifier
        self.fc1      = nn.Linear(out_2 * 4 * 4, number_of_classes)

    def forward(self, x):
        x = torch.relu(self.cnn1(x))
        x = self.maxpool1(x)
        x = torch.relu(self.cnn2(x))
        x = self.maxpool2(x)
        x = x.view(x.size(0), -1)   # flatten
        x = self.fc1(x)
        return x


# -----------------------------------------------------------------------
# CNN with Batch Normalization
# -----------------------------------------------------------------------
class CNN_batch(nn.Module):
    """
    Two-layer CNN with Batch Normalization after each conv and the FC layer.

    Benefits of BatchNorm:
    - Reduces internal covariate shift
    - Allows higher learning rates
    - Acts as regulariser (reduces need for dropout)
    - Faster convergence
    """

    def __init__(self, out_1=16, out_2=32, number_of_classes=10):
        super(CNN_batch, self).__init__()
        # Block 1
        self.cnn1      = nn.Conv2d(in_channels=1, out_channels=out_1, kernel_size=5, padding=2)
        self.conv1_bn  = nn.BatchNorm2d(out_1)
        self.maxpool1  = nn.MaxPool2d(kernel_size=2)
        # Block 2
        self.cnn2      = nn.Conv2d(in_channels=out_1, out_channels=out_2, kernel_size=5,
                                    stride=1, padding=2)
        self.conv2_bn  = nn.BatchNorm2d(out_2)
        self.maxpool2  = nn.MaxPool2d(kernel_size=2)
        # Classifier
        self.fc1       = nn.Linear(out_2 * 4 * 4, number_of_classes)
        self.bn_fc1    = nn.BatchNorm1d(number_of_classes)

    def forward(self, x):
        x = torch.relu(self.conv1_bn(self.cnn1(x)))
        x = self.maxpool1(x)
        x = torch.relu(self.conv2_bn(self.cnn2(x)))
        x = self.maxpool2(x)
        x = x.view(x.size(0), -1)
        x = self.bn_fc1(self.fc1(x))
        return x


# Instantiate
torch.manual_seed(0)
model_cnn   = CNN(out_1=16, out_2=32, number_of_classes=10)
model_batch = CNN_batch(out_1=16, out_2=32, number_of_classes=10)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"CNN parameter count:       {count_params(model_cnn):,}")
print(f"CNN_batch parameter count: {count_params(model_batch):,}")
print(f"\nCNN architecture:")
print(model_cnn)
print(f"\nCNN_batch architecture:")
print(model_batch)

---
## 3. Training Both Models

In [ ]:
def train_and_evaluate(model, train_loader, test_loader, dataset_val,
                       n_epochs=5, lr=0.1):
    """
    Train model with SGD + Cross Entropy Loss.
    Returns (cost_list, accuracy_list) per epoch.
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    N_test    = len(dataset_val)

    cost_list     = []
    accuracy_list = []
    start_time    = time.time()

    for epoch in range(n_epochs):
        model.train()
        cost = 0.0
        for x, y in train_loader:
            optimizer.zero_grad()
            z    = model(x)
            loss = criterion(z, y)
            loss.backward()
            optimizer.step()
            cost += loss.item()

        # Validation
        model.eval()
        correct = 0
        with torch.no_grad():
            for x_test, y_test in test_loader:
                z = model(x_test)
                _, yhat = torch.max(z.data, 1)
                correct += (yhat == y_test).sum().item()

        accuracy = correct / N_test
        cost_list.append(cost)
        accuracy_list.append(accuracy)
        elapsed = time.time() - start_time
        print(f"  Epoch {epoch+1}/{n_epochs} | "
              f"Cost: {cost:.2f} | "
              f"Acc: {accuracy*100:.2f}% | "
              f"Time: {elapsed:.1f}s")

    return cost_list, accuracy_list

In [ ]:
N_EPOCHS = 5
LR = 0.1

# Train CNN
print("Training CNN (standard)...")
torch.manual_seed(0)
model_cnn = CNN(out_1=16, out_2=32, number_of_classes=10)
cost_cnn, acc_cnn = train_and_evaluate(
    model_cnn, train_loader, test_loader, dataset_val,
    n_epochs=N_EPOCHS, lr=LR
)

print()

# Train CNN_batch
print("Training CNN_batch (with Batch Normalization)...")
torch.manual_seed(0)
model_batch = CNN_batch(out_1=16, out_2=32, number_of_classes=10)
cost_batch, acc_batch = train_and_evaluate(
    model_batch, train_loader, test_loader, dataset_val,
    n_epochs=N_EPOCHS, lr=LR
)

---
## 4. Results & Comparison

In [ ]:
# Plot cost and accuracy for both models
epochs = range(1, N_EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Cost
axes[0].plot(epochs, cost_cnn,   'b-o',  linewidth=2, markersize=7, label='CNN')
axes[0].plot(epochs, cost_batch, 'r-s',  linewidth=2, markersize=7, label='CNN + BatchNorm')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Training Cost (total loss)', fontsize=12)
axes[0].set_title('Training Cost per Epoch', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=12); axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs, [a*100 for a in acc_cnn],   'b-o',  linewidth=2, markersize=7,
             label=f'CNN (final: {acc_cnn[-1]*100:.2f}%)')
axes[1].plot(epochs, [a*100 for a in acc_batch], 'r-s',  linewidth=2, markersize=7,
             label=f'CNN+BN (final: {acc_batch[-1]*100:.2f}%)')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Validation Accuracy (%)', fontsize=12)
axes[1].set_title('Validation Accuracy per Epoch', fontsize=13, fontweight='bold')
axes[1].set_ylim(0, 100)
axes[1].legend(fontsize=12); axes[1].grid(True, alpha=0.3)

plt.suptitle('Fashion-MNIST: CNN vs CNN with Batch Normalization', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fashion_mnist_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fashion_mnist_results.png")

---
## 5. Error Analysis

In [ ]:
def get_misclassified(model, dataset_val, n=9):
    """Return up to n misclassified (image, true_label, pred_label) tuples."""
    model.eval()
    misclassified = []
    loader = DataLoader(dataset_val, batch_size=256, shuffle=False)
    idx = 0
    with torch.no_grad():
        for x, y in loader:
            z = model(x)
            _, preds = torch.max(z, 1)
            for i in range(len(y)):
                if preds[i].item() != y[i].item():
                    misclassified.append((x[i], y[i].item(), preds[i].item()))
                if len(misclassified) >= n:
                    return misclassified
            idx += len(y)
    return misclassified


# Show misclassified examples from the better model
best_model = model_batch if acc_batch[-1] >= acc_cnn[-1] else model_cnn
best_name  = 'CNN+BatchNorm' if acc_batch[-1] >= acc_cnn[-1] else 'CNN'
errors = get_misclassified(best_model, dataset_val, n=9)

if errors:
    fig, axes = plt.subplots(3, 3, figsize=(9, 9))
    for ax, (img, true_lbl, pred_lbl) in zip(axes.flatten(), errors):
        ax.imshow(img.numpy().squeeze(), cmap='gray')
        ax.set_title(f'True: {CLASS_NAMES[true_lbl]}\nPred: {CLASS_NAMES[pred_lbl]}',
                     fontsize=9, color='red')
        ax.axis('off')
    plt.suptitle(f'Misclassified Samples — {best_name}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('fashion_mnist_errors.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: fashion_mnist_errors.png")
else:
    print("No misclassified samples found — perfect accuracy!")

In [ ]:
# Per-class accuracy breakdown
def per_class_accuracy(model, dataset_val, n_classes=10):
    """Return per-class accuracy as a dict."""
    model.eval()
    correct = torch.zeros(n_classes)
    total   = torch.zeros(n_classes)
    loader  = DataLoader(dataset_val, batch_size=256, shuffle=False)
    with torch.no_grad():
        for x, y in loader:
            z = model(x)
            _, preds = torch.max(z, 1)
            for c in range(n_classes):
                mask = (y == c)
                correct[c] += (preds[mask] == y[mask]).sum().item()
                total[c]   += mask.sum().item()
    return {CLASS_NAMES[c]: (correct[c] / total[c]).item() * 100 for c in range(n_classes)}


class_acc_cnn   = per_class_accuracy(model_cnn,   dataset_val)
class_acc_batch = per_class_accuracy(model_batch, dataset_val)

print("Per-class accuracy:")
print(f"{'Class':<18} {'CNN':>8} {'CNN+BN':>10}")
print("-" * 40)
for cls in CLASS_NAMES:
    print(f"{cls:<18} {class_acc_cnn[cls]:>7.1f}%  {class_acc_batch[cls]:>7.1f}%")

# Bar chart
x = np.arange(len(CLASS_NAMES))
width = 0.35
fig, ax = plt.subplots(figsize=(13, 5))
bars1 = ax.bar(x - width/2, [class_acc_cnn[c] for c in CLASS_NAMES],
                width, label='CNN', color='steelblue', edgecolor='k')
bars2 = ax.bar(x + width/2, [class_acc_batch[c] for c in CLASS_NAMES],
                width, label='CNN+BatchNorm', color='tomato', edgecolor='k')
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=10)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Per-Class Accuracy: CNN vs CNN+BatchNorm', fontsize=13, fontweight='bold')
ax.legend(fontsize=12)
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('fashion_mnist_per_class.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fashion_mnist_per_class.png")

---
## 6. Summary

| Aspect | CNN | CNN + BatchNorm |
|---|---|---|
| Architecture | Conv→ReLU→Pool ×2, FC | Same + BN after each layer |
| Parameters | ~8,330 | ~8,490 (BN adds γ, β) |
| Training stability | Good | Better (reduced covariate shift) |
| Convergence speed | Standard | Often faster |
| Regularisation | None built-in | Mild (BN acts as regulariser) |

### Key Takeaways
1. **Batch Normalization** normalises layer inputs to zero mean and unit variance — this smooths the loss landscape and enables faster learning
2. **Padding=2 with kernel=5** preserves spatial dimensions after convolution (same padding)
3. **CrossEntropyLoss** is the correct loss for multi-class classification — it combines LogSoftmax + NLLLoss
4. Even 5 epochs is sufficient to achieve reasonable accuracy on Fashion-MNIST with a 2-layer CNN

In [ ]:
# Final summary table
print("=" * 55)
print("FINAL RESULTS: Fashion-MNIST CNN Comparison")
print("=" * 55)
print(f"{'Metric':<30} {'CNN':>10} {'CNN+BN':>10}")
print("-" * 55)
print(f"{'Parameters':<30} {count_params(model_cnn):>10,} {count_params(model_batch):>10,}")
print(f"{'Final Val Accuracy':<30} {acc_cnn[-1]*100:>9.2f}% {acc_batch[-1]*100:>9.2f}%")
print(f"{'Final Training Cost':<30} {cost_cnn[-1]:>10.2f} {cost_batch[-1]:>10.2f}")
print(f"{'Epoch 1 Accuracy':<30} {acc_cnn[0]*100:>9.2f}% {acc_batch[0]*100:>9.2f}%")
winner = 'CNN+BatchNorm' if acc_batch[-1] >= acc_cnn[-1] else 'CNN'
print(f"\n✓ Better model: {winner}")

---
<a id="consolidated-summary"></a>

---

# Consolidated Summary

## Key Formulas

| Concept | Formula | Notes |
|---|---|---|
| Conv output (basic) | $M_{new} = M - K + 1$ | No stride or padding |
| Conv output (general) | $M_{new} = \lfloor(M + 2p - K)/s\rfloor + 1$ | Stride $s$, padding $p$ |
| Same padding | $p = (K - 1) / 2$ | Output size = Input size (stride=1) |
| MaxPool output | $M_{new} = \lfloor(M - K)/s\rfloor + 1$ | Same formula as conv |
| ReLU | $f(x) = \max(0, x)$ | Element-wise, output same shape as input |
| BCE Loss | $\mathcal{L} = -\frac{1}{N}\sum[y\log\hat{y} + (1-y)\log(1-\hat{y})]$ | Binary classification |
| Cross Entropy | $\mathcal{L} = -\sum y_c \log\hat{y}_c$ | Multi-class classification |

## Architecture Comparison: CNN vs CNN + BatchNorm

| Layer | CNN | CNN + BatchNorm |
|---|---|---|
| Conv Block 1 | Conv2d → ReLU → MaxPool | Conv2d → **BatchNorm2d** → ReLU → MaxPool |
| Conv Block 2 | Conv2d → ReLU → MaxPool | Conv2d → **BatchNorm2d** → ReLU → MaxPool |
| Classifier | Linear(512→10) | Linear(512→10) → **BatchNorm1d** |

## Spatial Dimension Tracking (16×16 input)

```
Input      →  1 × 16 × 16
Conv1(k=5,p=2) →  16 × 16 × 16  (same padding)
MaxPool(k=2)   →  16 ×  8 ×  8
Conv2(k=5,p=2) →  32 ×  8 ×  8  (same padding)
MaxPool(k=2)   →  32 ×  4 ×  4
Flatten        →  512
Linear         →  10  (class logits)
```

## Key Takeaways

| Lab | Core Lesson |
|---|---|
| 1 — Convolution | Output size shrinks with larger kernels; padding recovers it; stride trades resolution for speed |
| 2 — Activation & Pooling | ReLU enables non-linearity; MaxPool reduces spatial size while retaining dominant features |
| 3 — Initialization | Bad weight initialization saturates sigmoid → near-zero gradients → model fails to learn |
| 4 — Batch Normalization | BatchNorm normalises layer inputs across the batch → faster convergence, more stable training |

---

*IBM AI Engineering Professional Certificate — Deep Neural Networks with PyTorch*  
*Author: Jack Pumpuni Frimpong-Manso | GitHub: [github.com/pumpuni07](https://github.com/pumpuni07)*